In [1]:
import sys

sys.path.append("..")

In [2]:
from pathlib import Path

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.datasets import (
    MultimodalCollator,
    MultimodalDataset,
)
from src.models import (
    ImageEncoder,
    MultimodalClassifier,
    TextEncoder,
)
from src.preprocessing import (
    ImageProcessor,
    TextProcessor,
)
from src.utils.config import load_config
from src.utils.device import get_device

PROJECT_ROOT = Path.cwd().parent

config = load_config(
    PROJECT_ROOT / "configs" / "base.yaml"
)

device = get_device(
    config["training"]["device"]
)

print("Device:", device)

c:\Users\AYUSH PRASAD\Desktop\Multimodal Misinformation Detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Loading Datasets & Processors

In [3]:
processed_dir = (
    PROJECT_ROOT
    / config["paths"]["processed_dir"]
)

train_df = pd.read_csv(
    processed_dir / "train.csv"
)

validate_df = pd.read_csv(
    processed_dir / "validate.csv"
)

text_config = config["models"]["text"]

text_processor = TextProcessor(
    model_name=text_config["name"],
    max_length=text_config["max_length"],
)

image_processor = ImageProcessor()

train_dataset = MultimodalDataset(
    train_df,
    processed_dir,
    image_processor,
)

collator = MultimodalCollator(
    text_processor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["training"]["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=collator,
)

In [4]:
batch = next(iter(train_loader))

print("Input IDs:", batch["input_ids"].shape)
print("Attention mask:", batch["attention_mask"].shape)
print("Images:", batch["images"].shape)
print("Labels:", batch["labels"].shape)

Input IDs: torch.Size([16, 18])
Attention mask: torch.Size([16, 18])
Images: torch.Size([16, 3, 224, 224])
Labels: torch.Size([16])


## Initializing the complete Fusion model

In [5]:
text_encoder = TextEncoder(
    model_name=text_config["name"],
    freeze=True,
).to(device)

image_encoder = ImageEncoder(
    freeze=True
).to(device)

fusion_config = config["models"]["fusion"]

model = MultimodalClassifier(
    text_encoder=text_encoder,
    image_encoder=image_encoder,
    hidden_dim=fusion_config["hidden_dim"],
    dropout=fusion_config["dropout"],
    num_classes=config["project"]["num_classes"],
    freeze_backbones=True,
).to(device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 25009.27it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Testing

In [6]:
criterion = nn.CrossEntropyLoss()

model.train()

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
images = batch["images"].to(device)
labels = batch["labels"].to(device)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
):
    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        images=images,
    )

    loss = criterion(
        logits,
        labels,
    )

loss.backward()

print("Logits:", logits.shape)
print("Loss:", loss.item())

Logits: torch.Size([16, 2])
Loss: 0.6637725830078125


In [7]:
trainable_with_grad = sum(
    parameter.grad is not None
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    "Trainable parameters with gradients:",
    trainable_with_grad,
)

Trainable parameters with gradients: 4
